### Torch CUDA and Imports

In [1]:
!pip install transformers==4.44.0 datasets scikit-learn accelerate -q
!pip install torch --index-url https://download.pytorch.org/whl/cu121 -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 70.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 106.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 89.3 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())      # must be True
print(torch.cuda.device_count())      # should be 2
print(torch.cuda.get_device_name(0))  # Tesla T4

2.10.0+cu128
True
1
Tesla T4


### Train / Fine-Tune on MultiNLI

In [4]:
"""
train_multinli.py  —  v19 (Kaggle)
Fine-tune DeBERTa-v3-base on MultiNLI.

ONLY changes from original v4:
  - OUTPUT_DIR path updated for Kaggle
  - BEST_MODEL save added at end
"""

import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    TrainerCallback,
)
from sklearn.metrics import f1_score

# ── Config ────────────────────────────────────────────────────────────────────

MODEL_NAME   = "microsoft/deberta-v3-base"
OUTPUT_DIR   = "/kaggle/working/multinli_checkpoint"
BEST_MODEL   = "/kaggle/working/multinli_checkpoint/buss305_multinli_bestmodel"
MAX_LENGTH   = 256
BATCH_SIZE   = 16
EPOCHS       = 3
LR           = 1e-5
WARMUP_STEPS = 1000

LABEL2ID = {"entailment": 0, "neutral": 1, "contradiction": 2}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}

# ── Load ──────────────────────────────────────────────────────────────────────

print("Loading MultiNLI...")
raw = load_dataset("multi_nli", split={
    "train":      "train",
    "validation": "validation_matched",
})
print(f"  Train : {len(raw['train'])}  |  Val : {len(raw['validation'])}")

# ── Tokenise ──────────────────────────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    enc = tokenizer(
        batch["premise"],
        batch["hypothesis"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )
    enc["labels"] = batch["label"]
    enc.pop("token_type_ids", None)
    return enc

print("Tokenising...")
dataset = raw.map(
    tokenize,
    batched=True,
    batch_size=512,
    remove_columns=raw["train"].column_names,
)
dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print("  Columns:", dataset["train"].column_names)

# ── Model ─────────────────────────────────────────────────────────────────────

print("Loading model...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)

# ── Metrics ───────────────────────────────────────────────────────────────────

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds    = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    accuracy = (preds == labels).mean()
    return {"macro_f1": macro_f1, "accuracy": accuracy}

# ── Grad norm + loss callback ─────────────────────────────────────────────────

class LogCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None:
            return
        loss      = logs.get("loss", None)
        grad_norm = logs.get("grad_norm", None)
        epoch     = logs.get("epoch", None)
        if loss is not None:
            parts = {}
            if epoch     is not None: parts["epoch"]     = round(epoch, 2)
            if loss      is not None: parts["loss"]      = round(loss, 4)
            if grad_norm is not None: parts["grad_norm"] = round(grad_norm, 4)
            print(f"  {parts}")

# ── Training args ─────────────────────────────────────────────────────────────

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=WARMUP_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=1000,
    log_level="error",
    log_level_replica="error",
    fp16=True,
    bf16=False,
    report_to="none",
)

# ── Train ─────────────────────────────────────────────────────────────────────

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[LogCallback()],
)

print("Starting training...")
trainer.train()

# ── Save ──────────────────────────────────────────────────────────────────────

trainer.save_model(BEST_MODEL)
tokenizer.save_pretrained(BEST_MODEL)
print(f"\nBest model saved to {BEST_MODEL}")
print(f"All checkpoints in  {OUTPUT_DIR}")
print("\n✓ train_multinli.py complete.")

2026-06-10 12:44:49.458505: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781095489.640218      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781095489.696623      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781095490.152421      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781095490.152459      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781095490.152461      58 computation_placer.cc:177] computation placer alr

Loading MultiNLI...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/214M [00:00<?, ?B/s]

data/validation_matched-00000-of-00001.p(…):   0%|          | 0.00/4.94M [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Starting training...


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,0.307100,0.286303,0.896118,0.896485
2,0.236300,0.314386,0.903535,0.903719
3,0.177300,0.374078,0.904325,0.904636


  {'epoch': 0.04, 'loss': 0.8155, 'grad_norm': 15.4401}
  {'epoch': 0.08, 'loss': 0.4719, 'grad_norm': 12.2043}
  {'epoch': 0.12, 'loss': 0.4227, 'grad_norm': 12.041}
  {'epoch': 0.16, 'loss': 0.3931, 'grad_norm': 8.4221}
  {'epoch': 0.2, 'loss': 0.3839, 'grad_norm': 6.79}
  {'epoch': 0.24, 'loss': 0.369, 'grad_norm': 3.8098}
  {'epoch': 0.29, 'loss': 0.3652, 'grad_norm': 11.1214}
  {'epoch': 0.33, 'loss': 0.3472, 'grad_norm': 10.0309}
  {'epoch': 0.37, 'loss': 0.3537, 'grad_norm': 5.7777}
  {'epoch': 0.41, 'loss': 0.3443, 'grad_norm': 23.6511}
  {'epoch': 0.45, 'loss': 0.3501, 'grad_norm': 13.6922}
  {'epoch': 0.49, 'loss': 0.3408, 'grad_norm': 1.5442}
  {'epoch': 0.53, 'loss': 0.3306, 'grad_norm': 9.5834}
  {'epoch': 0.57, 'loss': 0.3289, 'grad_norm': 2.7401}
  {'epoch': 0.61, 'loss': 0.3245, 'grad_norm': 5.9546}
  {'epoch': 0.65, 'loss': 0.329, 'grad_norm': 9.7607}
  {'epoch': 0.69, 'loss': 0.3291, 'grad_norm': 11.2098}
  {'epoch': 0.73, 'loss': 0.3269, 'grad_norm': 8.4173}
  {'epoc

### Download from Working Directory

In [5]:
import shutil
shutil.make_archive('/kaggle/working/multinli_bestmodel', 'zip',
                    '/kaggle/working/multinli_checkpoint/buss305_multinli_bestmodel')
print("Done")

Done
